In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Load dataset
df = pd.read_csv("new_dataset.csv")

# Convert Volumes
def convert_vol(x):
    if isinstance(x, str):
        x = x.replace(",", "")
        if "K" in x:
            return float(x.replace("K", "")) * 1_000
        elif "M" in x:
            return float(x.replace("M", "")) * 1_000_000
        else:
            return float(x)
    return x

df["Vol."] = df["Vol."].apply(convert_vol)

# Clean Change %
df["Change %"] = df["Change %"].astype(str).str.replace("%", "").astype(float)

# Target (Next-day direction)
df["Target"] = (df["Price"].shift(-1) > df["Price"]).astype(int)
df = df[:-1]

# Technical indicators
df["Return"] = df["Price"].pct_change()
df["SMA_5"] = df["Price"].rolling(5).mean()
df["SMA_10"] = df["Price"].rolling(10).mean()
df["EMA_10"] = df["Price"].ewm(span=10, adjust=False).mean()
df["Volatility"] = df["Price"].rolling(5).std()
df["Price_Lag1"] = df["Price"].shift(1)

delta = df["Price"].diff()
gain = np.where(delta > 0, delta, 0)
loss = np.where(delta < 0, -delta, 0)
avg_gain = pd.Series(gain).rolling(14).mean()
avg_loss = pd.Series(loss).rolling(14).mean()
rs = avg_gain / (avg_loss + 1e-10)
df["RSI"] = 100 - (100 / (1 + rs))

ema12 = df["Price"].ewm(span=12, adjust=False).mean()
ema26 = df["Price"].ewm(span=26, adjust=False).mean()
df["MACD"] = ema12 - ema26
df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

df["BB_Mid"] = df["Price"].rolling(20).mean()
df["BB_Upper"] = df["BB_Mid"] + 2 * df["Price"].rolling(20).std()
df["BB_Lower"] = df["BB_Mid"] - 2 * df["Price"].rolling(20).std()

df = df.dropna()

# Features & Labels
features = [
    "Open","High","Low","Vol.","Return","SMA_5","SMA_10","EMA_10",
    "Volatility","Price_Lag1","RSI","MACD","MACD_Signal",
    "BB_Mid","BB_Upper","BB_Lower"
]

X = df[features]
y = df["Target"]

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split (time-series aware)
split = int(len(X_scaled) * 0.8)
X_train, X_test = X_scaled[:split], X_scaled[split:]
y_train, y_test = y[:split], y[split:]

# Logistic Regression model (best performer)
model = LogisticRegression(
    max_iter=3000,
    C=2.5,
    solver="lbfgs",
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_test)

print("\n📊 Logistic Regression Results")
print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("Precision:", round(precision_score(y_test, preds), 4))
print("Recall:", round(recall_score(y_test, preds), 4))
print("F1-Score:", round(f1_score(y_test, preds), 4))
print(classification_report(y_test, preds))

# Predict next day
latest_data = X_scaled[-1].reshape(1, -1)
tomorrow = model.predict(latest_data)[0]

print("\n🔮 Prediction for Tomorrow:")
print("Logistic Regression:", "UP" if tomorrow == 1 else "DOWN")

FileNotFoundError: [Errno 2] No such file or directory: 'new_dataset.csv'